In [ ]:
# 2026-11-11
import pandas as pd
from csae_pyutils import gsheet_to_df
from tqdm import tqdm
from django.core.exceptions import ObjectDoesNotExist
from apis_core.apis_metainfo.models import Collection, Uri, TempEntityClass
from apis_core.apis_entities.models import Person, Work
from apis_core.apis_vocabularies.models import WorkType, PersonWorkRelation
from apis_core.apis_relations.models import PersonWork


In [ ]:
col = Collection.objects.get(id=122)
domain = "spielplan-oper_1869–1945"

In [ ]:
df = gsheet_to_df("1FqrdhZY8a8BumiNaqkUHl5cxCR8l2mYL19hMRmw7vak")
df.head(3)

In [ ]:
work_kinds = {}
persons = {}
composer = PersonWorkRelation.objects.get(id=1217)
librettist = PersonWorkRelation.objects.get(id=5070)
for i, row in tqdm(df.iterrows(), total=len(df)):
    pmb_uri, _ = Uri.objects.get_or_create(uri=row["uri"], domain=domain)
    entity = pmb_uri.entity
    if entity:
        work = Work.objects.get(id=entity.id)
        pass
    else:
        item = {
            "name": row["title"][:254] if pd.notna(row["title"]) else "",
        }
        
        entity = Work.objects.create(**item)
    try:
        entity.start_date_written = str(int(row["date"]))
    except:
        pass

    entity.save()
    work = Work.objects.get(id=entity.id)
    if pd.notna(row["work_kind"]):
        try:
            type_of_work = work_kinds[row["work_kind"]]
        except KeyError:
            type_of_work = WorkType.objects.get_or_create(name=row["work_kind"])[0]
            work_kinds[row["work_kind"]] = type_of_work
        work.kind = type_of_work
    work.save()
    pmb_uri.entity = entity
    pmb_uri.save()
    entity.collection.add(col)
    if pd.notna(row["composers"]):
        for x in row["composers"].split(" | "):
            stub, person_id = x.split(":")
            if stub == "opera":
                uri = f"https://www.mdw.ac.at/imi/operapolitics/spielplan-wiener-oper/web/person/{person_id}"
            else:
                uri = f"https://pmb.acdh.oeaw.ac.at/entity/{person_id}/"
            try:
                person = persons[uri]
            except KeyError:
                person_id = Uri.objects.get(uri=uri).entity
                person = Person.objects.get(id=person_id.id)
                persons[uri] = person
            PersonWork.objects.get_or_create(related_person=person, related_work=work, relation_type=composer)   
    if pd.notna(row["librettists"]):
        for x in row["librettists"].split(" | "):
            stub, person_id = x.split(":")
            if stub == "opera":
                uri = f"https://www.mdw.ac.at/imi/operapolitics/spielplan-wiener-oper/web/person/{person_id}"
            else:
                uri = f"https://pmb.acdh.oeaw.ac.at/entity/{person_id}/"
            try:
                person = persons[uri]
            except KeyError:
                person_id = Uri.objects.get(uri=uri).entity
                person = Person.objects.get(id=person_id.id)
                persons[uri] = person
            PersonWork.objects.get_or_create(related_person=person, related_work=work, relation_type=librettist)    
                
            